Genie helped me to find the URL path to the external storage.
It run
`SHOW EXTERNAL LOCATIONS`

In [0]:
%sql
show external locations

In [0]:
%python

location_name = "demoworkspace"
subdir = "external_storage"

locations = spark.sql("SHOW EXTERNAL LOCATIONS").collect()

for location in locations:
  if location.name == location_name:
    dbutils.widgets.text("external_location", f"{location.url.rstrip('/')}/{subdir}")
    break
else:
  raise ValueError(f"External location '{location_name}' not found")

external_location = dbutils.widgets.get("external_location")
print(external_location)

Managed Tables

In [0]:
%sql
create table managed_default
    (width INT, length INT, height INT);

insert into managed_default
    values (3 INT, 2 INT, 1 INT)

In [0]:
%sql
describe extended managed_default

External Table

In [0]:
%sql
create or replace table external_default
    (width INT, length INT, height INT)
location '${external_location}/external_default';

insert into external_default
    values (3 INT, 2 INT, 1 INT)

In [0]:
%sql
select * from external_default

Drops

In [0]:
%sql
drop table managed_default

In [0]:
%fs ls 'dbfs:/mnt/demo/external_default'

In [0]:
%sql
drop table external_default

In [0]:
%python
display(dbutils.fs.ls(f'{external_location}/external_default'))

Schemas

In [0]:
%sql
create schema new_default

In [0]:
%sql
describe database extended new_default

In [0]:
%sql
use new_default;

create table managed_new_default
    (width int, lenght int, height int);

insert into managed_new_default
values (3 int, 2 int, 1 int);

-------------------------------------------

create table external_new_default
    (width int, length int, height int)
    location '${external_location}/external_new_default';

insert into external_new_default
values (3 int, 2 int, 1 int);

In [0]:
%sql
describe extended managed_new_default;

In [0]:
%sql
describe extended external_new_default;

In [0]:
%sql
drop table managed_new_default;
drop table external_new_default;

In [0]:
%sql
create schema custom
managed location '${external_location}/custom'

With MANAGED LOCATION, you are explicitly telling Unity Catalog where the schema’s managed objects should live in cloud storage. That means any managed table or managed volume created in that schema will use that schema-level path. It gives you tighter control over storage placement, isolation, and organization.

With no MANAGED LOCATION, the schema still works, but it inherits its managed storage from a higher level, usually the catalog’s managed location, or otherwise the metastore default. So the main difference is not behavior of the schema itself, but who decides the storage path for managed objects created inside it.

A useful rule of thumb:

Use MANAGED LOCATION when you want this schema’s managed data in a specific path.
Omit it when inherited default storage is fine.
It affects managed tables/volumes, not external tables, because external tables still use their own LOCATION at table creation time.

In [0]:
%sql
describe database extended custom

In [0]:
%sql
USE SCHEMA custom;

CREATE TABLE managed_custom
  (width INT, length INT, height INT);
  
INSERT INTO managed_custom
VALUES (3 INT, 2 INT, 1 INT);

-----------------------------------

CREATE TABLE external_custom
  (width INT, length INT, height INT)
LOCATION '${external_location}/external_custom';
  
INSERT INTO external_custom
VALUES (3 INT, 2 INT, 1 INT);

In [0]:
%sql
DESCRIBE EXTENDED managed_custom

In [0]:
%sql
DESCRIBE EXTENDED external_custom

In [0]:
%sql
DROP TABLE managed_custom;
DROP TABLE external_custom;

In [0]:
%python
display(dbutils.fs.ls(f'{external_location}/custom'))